In [1]:
import scanpy as sc
import pandas as pd
import os
import glob
import re
from pathlib import Path
import infercnvpy as cnv
import matplotlib.pyplot as plt
import numpy as np
import squidpy as sq
import insitucnv as icv
from insitucnv.analysis import find_optimal_clustering


input_dir = '/nfs/team283/DKFZ_Winkler_Xenium/DKFZ_SANGER/'
output_dir = '/lustre/scratch124/cellgen/bayraktar/gd11/projects/Winkler_Xenium_2026/output/'
#├── 20251112__100223__062_B320_Run01_2025-11-12
#├── 20260227__085858__062_B320_Run02_2026-02-27

/software/conda/users/gd11/insitucnv_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/software/conda/users/gd11/insitucnv_env/lib/python3.11/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/software/conda/users/gd11/insitucnv_env/lib/python3.11/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from 

In [ ]:
tumour_files = glob.glob(output_dir+'/CNV_results/*_CN_malignant.h5ad')

malignant_clusters = {}
for file in tumour_files:
    batch = re.sub('^.*/(.*?)_CN_malignant.h5ad', '\\1', file)
    outfile = os.path.join(output_dir, 'CNV_results',
                        '{}_malignant_assignment.csv'.format(batch))
    print(batch)
    
    adata = sc.read_h5ad(file)
    adata.obs[['malignant_leiden']].to_csv(outfile)

In [ ]:
# OSI43
# OSI39
# OSI44
# OSI36
# OSI30
# OSI28

In [ ]:
tumour_files = glob.glob(output_dir+'/CNV_results/*_CN.h5ad')

query = 'OSI30'

malignant_clusters = {}
for file in [i for i in tumour_files if query in i]:
    batch = re.sub('^.*/(.*?)_CN.h5ad', '\\1', file)
    adata = sc.read_h5ad(file)

    adata.obsm["X_spatial"] = adata.obs[
        ["cell_centroid_x", "cell_centroid_y"]
    ].to_numpy(dtype="float32")

    
    malignant_assignment = pd.read_csv(os.path.join(output_dir, 'CNV_results',
                                                   query+'_malignant_assignment.csv'),
                                      index_col = 0)
    malignant_assignment['malignant_leiden'] = ['clone_'+str(i) for i in malignant_assignment['malignant_leiden']]

    adata.obs = adata.obs.reset_index().merge(
        malignant_assignment.reset_index(), 
        on = 'index', how = 'left').set_index('index')
    adata.obs['cluster_assignment'] = np.where(
        adata.obs['malignant_leiden'].isnull(), 'diploid',
        adata.obs['malignant_leiden']
    )
    

In [ ]:
sq.pl.spatial_scatter(
    adata,
    #sp_ad[sp_ad.obs['cell_type_probability']>0.4],
    spatial_key="X_spatial",
    color="cluster_assignment",
    img=False,
    shape=None,
    size = 2.5,
    figsize=(8, 8),

)